# Lab 1 — First contact: talking to a model, and measuring it

**Artificial Intelligence · 4th year · UIR · Pr. Hakim Hafidi**

| | |
|---|---|
| **Duration** | 2 hours, individual work |
| **You will learn** | how to call a model from Colab without exposing your key · how to read what a call costs (tokens, latency, money) · how to compare two models on a task from your own field |
| **Deliverable** | this notebook, executed, + the `lab01_<your id>.json` file it generates, both submitted on connect.uir.ac.ma |
| **Grading** | 4 points: it runs (1) · measurements are correct (1) · interpretation (1) · critical analysis (1) |

> **Rule for the whole semester:** never paste an API key into a notebook cell.
> You will use Colab Secrets instead — step 1 below.

---
## Step 0 — Get your API key (10 minutes)

1. Go to **https://aistudio.google.com/apikey** and sign in with your Google account.
2. Click **Create API key**. No credit card is needed. Copy the key.
3. In this notebook, open the **🔑 key icon** in the left sidebar → **+ Add new secret**:
   - **Name:** `GEMINI_API_KEY`
   - **Value:** paste your key
   - Switch **Notebook access** ON.
4. Run the two cells below.

*If anything fails, raise your hand — do not spend ten minutes alone on it.*

In [ ]:
# Install the libraries this lab needs (~30 seconds)
%pip install -q google-genai pandas
print("done")

In [ ]:
#@title ⚙️ Toolbox — run this cell once, then fold it { display-mode: "form" }
# =============================================================================
#  AI course · UIR · shared lab toolbox  (v1)
#  You do not need to modify this cell. Read it once — you will use ask() all
#  semester. It handles: the API key, caching, rate limits, and measurement.
# =============================================================================
import os, re, json, time, hashlib, pathlib, textwrap
from dataclasses import dataclass, asdict

# --- Models available on the Gemini free tier --------------------------------
# Names change every few months. If a call fails with "model not found",
# check https://ai.google.dev/gemini-api/docs/models and update these two lines.
MODEL_FAST   = "gemini-2.5-flash-lite"   # cheapest, highest daily quota
MODEL_STRONG = "gemini-2.5-flash"        # better, lower quota

# Public prices, USD per 1M tokens — used to *simulate* what your calls cost.
PRICES = {
    "gemini-2.5-flash-lite": (0.10, 0.40),
    "gemini-2.5-flash":      (0.30, 2.50),
}

CACHE_DIR = pathlib.Path("/content/.ai_cache"); CACHE_DIR.mkdir(exist_ok=True, parents=True)
CALL_LOG  = []          # every call made in this session
USE_LOCAL_FALLBACK = False   # set to True only if your API key does not work

@dataclass
class Answer:
    text: str; model: str; temperature: float
    in_tokens: int; out_tokens: int; latency_s: float; cost_usd: float
    cached: bool = False
    def __str__(self): return self.text
    def __repr__(self): return self.text

def _api_key():
    """Read the key from Colab Secrets (preferred) or an env variable."""
    try:
        from google.colab import userdata
        k = userdata.get("GEMINI_API_KEY")
        if k: return k.strip()
    except Exception:
        pass
    return (os.environ.get("GEMINI_API_KEY") or "").strip()

_client = None
def _client_once():
    global _client
    if _client is None:
        from google import genai
        key = _api_key()
        if not key:
            raise RuntimeError(
                "No API key found. Add it in Colab: 🔑 (left sidebar) → + New secret →\n"
                "  Name: GEMINI_API_KEY   Value: <your key>   → enable 'Notebook access'."
            )
        _client = genai.Client(api_key=key)
    return _client

# --- Emergency fallback: a small open model running locally in Colab ---------
_local = None
def _local_once():
    global _local
    if _local is None:
        print("⚠️  Loading a small local model (slow, low quality — emergency use only)…")
        from transformers import pipeline
        _local = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct",
                          max_new_tokens=300, do_sample=True)
    return _local

def _cache_path(payload):
    return CACHE_DIR / (hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:24] + ".json")

def ask(prompt, model=MODEL_FAST, temperature=0.0, max_tokens=800,
        system=None, use_cache=True, retries=5, verbose=False):
    """Send one prompt to a model and return an Answer.

    ask("Hello")                          -> fast model, temperature 0
    ask("Hello", model=MODEL_STRONG)      -> stronger model
    ask("Hello", temperature=1.5)         -> more random
    ask("Hello", use_cache=False)         -> force a real call (costs quota!)

    Results are cached on disk: re-running a cell with the same prompt and the
    same settings is free and instant. That is why your notebook can be
    re-executed from top to bottom without burning your daily quota.
    """
    key = {"p": prompt, "m": model, "t": temperature, "mt": max_tokens, "s": system}
    cp = _cache_path(key)
    if use_cache and cp.exists():
        d = json.loads(cp.read_text()); d["cached"] = True
        a = Answer(**d); CALL_LOG.append(asdict(a)); return a

    if USE_LOCAL_FALLBACK:
        t0 = time.time()
        out = _local_once()(prompt, temperature=max(temperature, 0.01))[0]["generated_text"]
        txt = out[len(prompt):].strip() if out.startswith(prompt) else out.strip()
        a = Answer(txt, "local-fallback", temperature, len(prompt)//4, len(txt)//4, time.time()-t0, 0.0)
        CALL_LOG.append(asdict(a)); return a

    from google.genai import types
    cfg = types.GenerateContentConfig(temperature=temperature, max_output_tokens=max_tokens,
                                      system_instruction=system)
    delay = 2.0
    for attempt in range(retries):
        try:
            t0 = time.time()
            r = _client_once().models.generate_content(model=model, contents=prompt, config=cfg)
            dt = time.time() - t0
            u  = getattr(r, "usage_metadata", None)
            ti = getattr(u, "prompt_token_count", 0) or 0
            to = getattr(u, "candidates_token_count", 0) or 0
            pi, po = PRICES.get(model, (0.0, 0.0))
            a = Answer((r.text or "").strip(), model, temperature, ti, to, round(dt, 2),
                       round(ti/1e6*pi + to/1e6*po, 6))
            cp.write_text(json.dumps({k: v for k, v in asdict(a).items() if k != "cached"}))
            CALL_LOG.append(asdict(a))
            if verbose: print(f"[{model}] {ti}→{to} tokens · {dt:.2f}s")
            return a
        except Exception as e:
            msg = str(e)
            if any(s in msg for s in ("429", "RESOURCE_EXHAUSTED", "quota", "503", "UNAVAILABLE")):
                print(f"⏳ rate limit or server busy — waiting {delay:.0f}s "
                      f"(attempt {attempt+1}/{retries}). Free tier is ~15 calls/minute.")
                time.sleep(delay); delay *= 2; continue
            raise
    raise RuntimeError("Still rate-limited after several retries. Wait a minute, or use the cache.")

def usage_report():
    """Print what this session has cost you so far."""
    import pandas as pd
    if not CALL_LOG: print("No calls yet."); return None
    df = pd.DataFrame(CALL_LOG)
    real = df[~df.cached]
    print(f"calls: {len(df)}  (real: {len(real)}, from cache: {int(df.cached.sum())})")
    print(f"tokens in/out: {int(real.in_tokens.sum())} / {int(real.out_tokens.sum())}")
    print(f"time spent waiting: {real.latency_s.sum():.1f}s    simulated cost: ${real.cost_usd.sum():.5f}")
    return df

def check_environment():
    """Verify that everything needed for this lab is available."""
    ok = True
    try:
        import google.genai; print("✅ google-genai installed")
    except ImportError:
        print("❌ google-genai missing — run the install cell above"); ok = False
    if _api_key(): print("✅ API key found")
    else: print("❌ API key not found — see the instructions above"); ok = False
    try:
        a = ask("Reply with exactly: OK", max_tokens=10)
        print(f"✅ model answered: {a.text[:40]!r}")
    except Exception as e:
        print(f"❌ call failed: {str(e)[:200]}"); ok = False
    print("\n" + ("🎉 You are ready." if ok else "⚠️  Fix the ❌ above, then re-run this cell."))
    return ok

def export_lab(lab_id, student_id, answers: dict, tables: dict = None, extra: dict = None):
    """Build the file you submit on connect.uir.ac.ma."""
    import pandas as pd
    student_id = str(student_id).strip()
    assert student_id and student_id.lower() not in ("", "xxxxx", "your_id"), \
        "Set STUDENT_ID to your real apogee number."
    payload = {"lab": lab_id, "student_id": student_id,
               "answers": {k: str(v).strip() for k, v in answers.items()},
               "tables": {k: (v.to_dict("records") if hasattr(v, "to_dict") else v)
                          for k, v in (tables or {}).items()},
               "extra": extra or {},
               "call_log": CALL_LOG,
               "created": time.strftime("%Y-%m-%d %H:%M")}
    name = f"{lab_id}_{student_id}.json"
    pathlib.Path(name).write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    empty = [k for k, v in payload["answers"].items() if len(v) < 40]
    print(f"📦 {name} written ({pathlib.Path(name).stat().st_size} bytes)")
    if empty: print(f"⚠️  These answers look too short: {empty}")
    else:     print("✅ All answers filled.")
    print("\nSubmit BOTH files on connect.uir.ac.ma:")
    print(f"   1. this notebook (File → Download → .ipynb)\n   2. {name}")
    return payload

print("✅ Toolbox loaded. Main function: ask(prompt, model=…, temperature=…)")
print(f"   Models: MODEL_FAST={MODEL_FAST!r}  MODEL_STRONG={MODEL_STRONG!r}")

---
## Step 1 — Check that everything works

The cell below sends one small prompt. If you see ✅ three times, you are ready.

In [ ]:
check_environment()

---
## Part A — The knobs (20 minutes)

`ask()` is the only function you need. Read its three lines of documentation:

```python
ask("Hello")                      # fast model, temperature 0
ask("Hello", model=MODEL_STRONG)  # stronger model
ask("Hello", temperature=1.5)     # more random
```

Run the next cell as it is, and look at what the model answered **and** at what it cost.

In [ ]:
a = ask("In one sentence, what is a language model?")
print(a.text)
print()
print(f"model      : {a.model}")
print(f"tokens     : {a.in_tokens} in → {a.out_tokens} out")
print(f"latency    : {a.latency_s} s")
print(f"cost       : ${a.cost_usd:.6f}   (simulated at public prices)")
print(f"from cache : {a.cached}")

### A.1 — Does the same question give the same answer?

Ask **the same question three times at temperature 0**, then **three times at temperature 1.5**.
Look at whether the answers are identical.

> ⚠️ `ask()` caches results: the same prompt with the same settings returns the *stored*
> answer instantly. To force real calls, pass `use_cache=False`.

In [ ]:
QUESTION = "Give me one original idea for a student project about AI in agriculture."

cold, hot = [], []
for i in range(3):
    # TODO (2 lines): call ask() with temperature=0.0, then with temperature=1.5.
    #                 Use use_cache=False so that each call is a real one.
    #                 Append the .text of each answer to `cold` and `hot`.
    cold.append(...)
    hot.append(...)

assert all(isinstance(t, str) for t in cold + hot),     "Fill the two TODO lines above: cold.append(ask(...).text)"

print("TEMPERATURE 0.0")
for t in cold: print(" -", t[:110].replace("\n", " "), "…")
print("\nTEMPERATURE 1.5")
for t in hot:  print(" -", t[:110].replace("\n", " "), "…")

print(f"\ndistinct answers at T=0.0 : {len(set(cold))}/3")
print(f"distinct answers at T=1.5 : {len(set(hot))}/3")

In [ ]:
assert len(cold) == 3 and len(hot) == 3, "You need 3 answers at each temperature."
assert all(isinstance(t, str) and len(t) > 20 for t in cold + hot), "Store the .text, not the Answer object."
print("✅ A.1 done — write down what you observed, you will need it in Part C.")

---
## Part B — Your task, two models (30 minutes)

Pick **one** task that is realistic for your own field, from these five starting points:

| | Task | Example |
|---|---|---|
| 1 | Summarize a technical text | a page of a standard, a paper abstract, a regulation |
| 2 | Extract information from a document | dates, amounts, names, parties in a contract |
| 3 | Explain a concept to a non-specialist | explain a notion of your speciality to a 15-year-old |
| 4 | Classify short messages | customer feedback, support tickets, survey answers |
| 5 | Write a short piece of code | a SQL query, a data-cleaning script |

Then write **one prompt** for it. Do not spend more than five minutes on the prompt — a
whole session (Lab 6) is devoted to making prompts good. Here we only need a prompt that works.

In [ ]:
# TODO (3–6 lines): describe your task and write your prompt.
#   MY_TASK  : one sentence, in English, saying what the task is.
#   MY_PROMPT: the actual text you send to the model. Include the input it must work on
#              (a paragraph you paste, a list of messages, a schema… ).

MY_TASK = "..."

MY_PROMPT = """
...
"""

assert len(MY_TASK) > 20 and len(MY_PROMPT) > 120, "Write a real task and a real prompt."
print(f"Task   : {MY_TASK}")
print(f"Prompt : {len(MY_PROMPT)} characters")

### B.1 — Run it on both models, three times each

The loop is written for you. You only fill in the two model names and the temperature.
This makes **6 real calls** — that is fine, the free tier allows about 15 per minute.

In [ ]:
import pandas as pd

rows = []
for model in [MODEL_FAST, MODEL_STRONG]:
    for run in range(3):
        # TODO (1 line): call ask() with MY_PROMPT, the current `model`,
        #                temperature=0.7, and use_cache=False.
        a = ...
        assert hasattr(a, "text"), "Fill the TODO line above: a = ask(MY_PROMPT, ...)"
        rows.append({"model": model, "run": run + 1, "answer": a.text,
                     "in_tokens": a.in_tokens, "out_tokens": a.out_tokens,
                     "latency_s": a.latency_s, "cost_usd": a.cost_usd})

results = pd.DataFrame(rows)
print(results[["model", "run", "in_tokens", "out_tokens", "latency_s", "cost_usd"]].to_string(index=False))

In [ ]:
# TODO (2–4 lines): build a summary table with, for each model, the MEAN of
#   out_tokens, latency_s and cost_usd over the 3 runs.
#   Hint: results.groupby("model")[[...]].mean().round(6)
#   Costs per call are tiny — keep 6 decimals, or you will read 0.0 everywhere.

summary = ...

# Given to you: what 1000 calls would cost — the number that actually matters.
summary["cost_per_1000_calls"] = (summary["cost_usd"] * 1000).round(2)

print(summary)

In [ ]:
# Read two answers side by side — one per model — and judge them yourself.
for model in [MODEL_FAST, MODEL_STRONG]:
    print("=" * 78)
    print(model)
    print("=" * 78)
    print(results[results.model == model].iloc[0].answer[:1200])
    print()

In [ ]:
assert len(results) == 6, "You need 3 runs on each of the 2 models."
assert hasattr(summary, "shape"), "summary is still the placeholder `...` — build the table."
assert len(summary) == 2, "summary must have exactly one row per model."
assert results.out_tokens.sum() > 0, "Token counts are zero — did the calls really run?"
print("✅ Part B done.")

---
## Part C — Tokens are not words (10 minutes)

One more measurement before the questions. The model does not read words, it reads
**tokens**. Let's see what that costs in different languages.

In [ ]:
SENTENCES = {
    "English": "Artificial intelligence is changing how engineers work every day.",
    "French":  "L'intelligence artificielle change la façon dont les ingénieurs travaillent chaque jour.",
    "Arabic":  "الذكاء الاصطناعي يغير طريقة عمل المهندسين كل يوم.",
}

rows = []
for lang, sentence in SENTENCES.items():
    # TODO (1 line): send the sentence to the model with max_tokens=5 and read a.in_tokens.
    #   We only care about how many tokens the INPUT takes, not about the answer.
    a = ...
    assert hasattr(a, "in_tokens"), "Fill the TODO line above: a = ask(sentence, max_tokens=5, ...)"
    rows.append({"language": lang, "characters": len(sentence),
                 "words": len(sentence.split()), "input_tokens": a.in_tokens})

tokens = pd.DataFrame(rows)
tokens["tokens_per_word"] = (tokens.input_tokens / tokens.words).round(2)
print(tokens.to_string(index=False))

In [ ]:
assert len(tokens) == 3 and tokens.input_tokens.min() > 0, "The three measurements must be real."
print("✅ Measurement done. Look at the last column before answering Q3.")

---
## Part D — Analysis (20 minutes)

Answer in the cell below, **in English, in your own words**, using your own numbers.
Two or three sentences each. This is where most of the grade is.

In [ ]:
STUDENT_ID = "XXXXX"   # TODO: your apogee number

# Q1 — Same prompt, same settings, run three times. Were the answers identical at
#      temperature 0? At 1.5? Quote two concrete differences you saw.
Q1 = """
...
"""

# Q2 — For YOUR task: which of the two models would you use, and why? Justify with at
#      least two of the five criteria from the course (quality, cost, latency, privacy, control)
#      and with your measured numbers (latency, cost per 1000 calls, answer length).
Q2 = """
...
"""

# Q3 — Look at the tokens_per_word column. Which language costs most per word, and by
#      roughly what factor? What practical consequence does that have for a Moroccan
#      company that wants to deploy an assistant in Arabic?
Q3 = """
...
"""

# Q4 — Name ONE thing about the quality of the answers that your method could NOT
#      measure. How would you measure it properly?
Q4 = """
...
"""

In [ ]:
answers = {"Q1": Q1, "Q2": Q2, "Q3": Q3, "Q4": Q4}
problems = []
if STUDENT_ID.strip().upper() in ("XXXXX", ""): problems.append("STUDENT_ID is not filled in")
for k, v in answers.items():
    if len(v.strip()) < 120: problems.append(f"{k} is too short (write 2–3 sentences)")
    if "..." in v:           problems.append(f"{k} still contains the placeholder")
if problems:
    print("⚠️  Not ready to submit:"); [print("   -", p) for p in problems]
else:
    print("✅ Everything looks complete. Run the export cell below.")

---
## Submit

Run the cell below, then upload **both** the notebook and the `.json` file on connect.uir.ac.ma.

In [ ]:
usage_report()
print()
export_lab("lab01", STUDENT_ID,
           answers={"Q1": Q1, "Q2": Q2, "Q3": Q3, "Q4": Q4},
           tables={"comparison": results.drop(columns=["answer"]),
                   "summary": summary.reset_index(),
                   "tokens": tokens},
           extra={"task": MY_TASK, "prompt": MY_PROMPT,
                  "distinct_T0": len(set(cold)), "distinct_T15": len(set(hot))})

---
## Going further (optional, not graded)

- Run your prompt at `temperature=0.0` and `temperature=2.0`, ten times each, and plot the
  number of distinct answers. Where does it stop being useful?
- Add a `system=` instruction to `ask()` (for example: *"You always answer in exactly three
  bullet points."*). Does it hold across the three runs?
- Take the answer you liked least and try to fix it **by changing the prompt only**.
  Keep both prompts — you will see in Lab 6 how to measure that improvement properly.